# Quest 47 — Q4-O · leakage-free residual CNN (`EXP-2026-001`)

## `ANALYZE_EXISTING_RUN` — 재학습 없이 기존 run을 읽어 보고서를 만든다

`OUT_DIR` 하나만 지정하면 이 노트북은 **학습을 전혀 하지 않고** Drive에 이미 저장된
run 번들(`result.json`, `manifest.json`, `predictions.npz`, `arms/<arm>/probs.npy`)만
읽어 표·그림·`report_summary.md`를 생성한다.

**이 개정은 presentation-only다.** 다음은 어느 것도 바뀌지 않는다:

| 바뀌지 않는 것 | 왜 |
|---|---|
| 기존 predictions / `probs.npy` | 읽기 전용으로만 연다 |
| arm 정의 (A~F) | `result.json`의 키를 그대로 사용 |
| fold, seed | `config.json` / `manifest.json`에서 읽는다 |
| metric, bootstrap | 재계산하지 않는다. CI는 `result.json`에서 그대로 읽는다 |
| gate와 NO-GO 판정 | `result.json`의 `gates`를 그대로 표시한다 |
| `result.json`의 측정값 | 마지막 셀에서 실행 전후 checksum을 비교해 증명한다 |

레코드 단위 그림(7·8번)만 `probs.npy`에서 값을 다시 계산하는데, 그 결과가
`result.json`의 `ach@k`를 재현하지 못하면 **그림을 그리지 않고 건너뛴다.**
재현되지 않은 숫자로는 아무것도 그리지 않는다.

> 그래프 축 라벨은 Colab의 한글 폰트 문제를 피하려고 영어를 쓴다.
> 해석·요약은 모두 한국어다.

## 0. 설정 — 여기만 바꾸면 된다

In [ ]:
# ── 유일한 필수 설정 ──────────────────────────────────────────────────────
ANALYZE_EXISTING_RUN = True

OUT_DIR = ("/content/drive/MyDrive/MedKOS/ecg-model/runs/"
           "20260806T0923_EXP-2026-001_q4o_leakage_free_residual_cnn")

# 그림을 run 폴더 안(figures/)이 아니라 다른 곳에 쓰고 싶으면 경로를 넣는다.
# None이면 <OUT_DIR>/figures 에 쓴다. 어느 쪽이든 측정 산출물은 건드리지 않는다.
REPORT_DIR = None

REPO_DIR = "/content/MedKOS"          # 이 저장소를 clone/mount 한 경로
MOUNT_DRIVE = True                    # Colab이 아니면 False

In [ ]:
import os, sys

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"[info] Drive 마운트 생략: {exc}")

for candidate in (os.path.join(REPO_DIR, "mit-bih"), "mit-bih", "."):
    if os.path.isfile(os.path.join(candidate, "q4o_leakage_free_residual.py")):
        sys.path.insert(0, os.path.abspath(candidate))
        break

import q4o_leakage_free_residual as q4o
print("module:", q4o.__file__)

## 1. run 번들 읽기 (읽기 전용) + 실행 **전** checksum

`result.json`을 포함한 모든 측정 산출물의 SHA-256을 먼저 찍어둔다.
마지막 셀에서 같은 값을 다시 찍어 보고 기능이 아무것도 건드리지 않았음을 확인한다.

In [ ]:
if not ANALYZE_EXISTING_RUN:
    raise SystemExit(
        "ANALYZE_EXISTING_RUN=False 입니다.\n"
        "이 노트북 개정은 보고(presentation) 전용이라 학습 경로를 포함하지 않습니다.\n"
        "Q4-O의 학습은 이미 끝난 run으로만 존재하며, 재학습은 과학적 결과를 바꾸므로\n"
        "이 노트북에서 수행하지 않습니다. OUT_DIR을 지정하고 True로 두세요."
    )

bundle = q4o.load_run(OUT_DIR)
checksums_before = q4o.bundle_checksums(OUT_DIR)

print(f"run           : {bundle.run_id}")
print(f"experiment    : {bundle.result['experiment_id']} / {bundle.result['arm_id']}")
print(f"seeds         : {bundle.seeds}  (n={bundle.n_seed})")
print(f"k-sweep       : {bundle.k_sweep}   ·  gate = +{bundle.min_gain:g}")
print(f"scorable recs : {bundle.manifest.get('n_record_scorable')} / {bundle.manifest.get('n_record_total')}")
print(f"predictions   : {list(bundle.predictions_keys) or '(없음)'}")
for w in bundle.load_warnings:
    print(f"[load] {w}")
print()
print("실행 전 checksum")
for name, digest in sorted(checksums_before.items()):
    print(f"  {digest[:16]}  {name}")

## 2. Executive Summary (한국어) — 가장 먼저 나오는 출력

In [ ]:
print(q4o.executive_summary_ko(bundle))

## 3. 전체 보고서 생성

표 · 그림 · `patient_delta.csv` · `arm_metrics.csv` · `report_summary.md`를 한 번에 만든다.
학습은 일어나지 않는다.

In [ ]:
result = q4o.analyze_existing_run(OUT_DIR, report_dir=REPORT_DIR, verbose=True)

FIG_DIR  = result["figures_dir"]
FIGURES  = result["figures"]
PER_REC  = result["per_record"]
HISTORY  = result["history"]

## 4. 표와 그림 — 각각 아래에 한국어 해석 2~3문장

해석 문장의 모든 숫자는 이 run에서 계산한 값이다. 문구가 데이터를 따라 움직인다.

In [ ]:
from IPython.display import display, Image, Markdown, HTML
import csv

def show(name, note=None):
    """그림/표 하나를 띄우고 그 아래에 한국어 해석을 붙인다."""
    path = FIGURES.get(name)
    display(Markdown(f"### {name}"))
    if not path or not os.path.isfile(path):
        display(Markdown(f"> **생성되지 않음.** {note or ''}"))
        text = q4o.interpretation_ko(bundle, name, PER_REC)
        if text:
            display(Markdown(f"*{text}*"))
        return
    if path.endswith(".png"):
        display(Image(filename=path))
    else:
        with open(path, encoding="utf-8") as fh:
            rows = list(csv.reader(fh))
        head, body = rows[0], rows[1:]
        html = ("<table><thead><tr>"
                + "".join(f"<th>{c}</th>" for c in head)
                + "</tr></thead><tbody>"
                + "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>"
                          for r in body[:20])
                + "</tbody></table>")
        display(HTML(html))
        if len(body) > 20:
            display(Markdown(f"*({len(body)}행 중 앞 20행. 전체는 `{os.path.basename(path)}`)*"))
    text = q4o.interpretation_ko(bundle, name, PER_REC)
    if text:
        display(Markdown(f"**해석 —** {text}"))

In [ ]:
show("arm_summary_table.png")

In [ ]:
show("arm_metrics.csv")

In [ ]:
show("primary_contrasts_zoom.png")

In [ ]:
show("reference_gap_separate.png")

In [ ]:
show("achievement_by_k.png")

In [ ]:
show("seed_effects.png")

In [ ]:
show("fold_training_diagnostics.png")

In [ ]:
show("patient_delta_waterfall.png", "레코드 단위 값이 result.json을 재현하지 못했습니다.")

In [ ]:
show("patient_delta.csv", "레코드 단위 값이 result.json을 재현하지 못했습니다.")

In [ ]:
show("metric_distribution.png", "레코드 단위 값이 result.json을 재현하지 못했습니다.")

## 5. 학습 이력 (training history)

이 run에는 epoch별 이력이 없다. **없는 데이터를 만들지 않는다** — 아래 셀은 그 사실을
출력하고 넘어간다. 향후 run은 `TrainingHistoryRecorder`를 학습 루프에 붙이면
`training_history.json`이 남고 이 자리에 `learning_curves.png`가 생긴다.

기록기는 **관찰만 한다**: optimizer 상태를 갖지 않고, 값을 되돌려주지 않으며,
checkpoint 선택에 관여하지 않는다. 따라서 붙여도 학습 연산과 best_epoch 선택이 바뀌지 않는다.

In [ ]:
print("training history 존재 여부 :", HISTORY["present"])
print("찾은 파일                 :", HISTORY["files"] or "(없음)")
print("기대 파일명               :", HISTORY["expected"])
print()
print(HISTORY["note"])

show("learning_curves.png", "이 run에는 epoch별 학습 이력이 없습니다.")

In [ ]:
# 향후 run에서 쓰는 방법 (여기서는 실행하지 않는다 — 학습을 하지 않으므로)
print(q4o.TrainingHistoryRecorder.__doc__)

## 6. 마지막 — 생성된 모든 그림과 `report_summary.md` 자동 표시

In [ ]:
display(Markdown("## 생성된 모든 그림"))
for name, path in FIGURES.items():
    if path and path.endswith(".png"):
        display(Markdown(f"**{name}**"))
        display(Image(filename=path))
        text = q4o.interpretation_ko(bundle, name, PER_REC)
        if text:
            display(Markdown(f"*{text}*"))
    elif not path:
        display(Markdown(f"**{name}** — 생성되지 않음"))

In [ ]:
display(Markdown("## report_summary.md"))
with open(result["report_summary"], encoding="utf-8") as fh:
    display(Markdown(fh.read()))

## 7. 무결성 — 보고 기능 실행 **후** checksum

여기서 `True`가 나오지 않으면 보고 기능이 측정 산출물을 건드린 것이므로 버그다.

In [ ]:
checksums_after = q4o.bundle_checksums(OUT_DIR)

print(f"{'파일':46s}  {'전':18s}  {'후':18s}  같음")
for name in sorted(set(checksums_before) | set(checksums_after)):
    b = checksums_before.get(name, "")
    a = checksums_after.get(name, "")
    print(f"{name:46s}  {b[:16]:18s}  {a[:16]:18s}  {b == a}")

assert checksums_before["result.json"] == checksums_after["result.json"], \
    "result.json이 변경되었습니다 — 보고 기능은 측정값을 건드리면 안 됩니다."
assert checksums_before == checksums_after, "측정 산출물이 변경되었습니다."
print()
print("OK — result.json을 포함한 모든 측정 산출물의 checksum이 실행 전후 동일합니다.")
print(f"판정은 그대로: {bundle.verdict}")